In [36]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../../../data/processed/wrapper_results")

def load_rfe_rf():
    df = pd.read_csv(DATA_DIR / "rfe_random_forest_results.csv")
    df = df.rename(columns={"features": "selected_features"})
    df["method"] = "RFE"; df["model"] = "Random Forest"
    df["mae"] = pd.NA; df["r2"] = pd.NA
    return df[["method","model","n_features","selected_features","rmse","mae","r2"]]

def load_rfe_xgb():
    df = pd.read_csv(DATA_DIR / "rfe_xgboost_results.csv")
    df = df.rename(columns={"rfe_n_features": "n_features"})
    df["method"] = "RFE"; df["model"] = "XGBoost"
    return df[["method","model","n_features","selected_features","rmse","mae","r2"]]

def load_sfs_rf():
    df = pd.read_csv(DATA_DIR / "sfs_random_forest_results.csv")
    df = df.rename(columns={"n_features_to_select":"n_features","RMSE":"rmse","MAE":"mae","R2":"r2"})
    df["method"] = "SFS"; df["model"] = "Random Forest"
    return df[["method","model","n_features","selected_features","rmse","mae","r2"]]

def load_sfs_xgb():
    df = pd.read_csv(DATA_DIR / "sfs_xgboost_results.csv")
    df = df.rename(columns={"n_features_to_select":"n_features","RMSE":"rmse","MAE":"mae","R2":"r2"})
    df["method"] = "SFS"; df["model"] = "XGBoost"
    return df[["method","model","n_features","selected_features","rmse","mae","r2"]]

comparison = pd.concat([load_rfe_rf(), load_rfe_xgb(), load_sfs_rf(), load_sfs_xgb()], ignore_index=True)
comparison_sorted = comparison.sort_values("rmse").reset_index(drop=True)

best_results = (
    comparison_sorted
    .loc[comparison_sorted.groupby(["method", "model"])["rmse"].idxmin()]
    .sort_values("rmse")
    .reset_index(drop=True)
)

print("=== Wrapper별 최적 성능 (RMSE 기준) ===\n")
for _, row in best_results.iterrows():
    line = f"{row['method']} + {row['model']}: features={int(row['n_features'])}, RMSE={row['rmse']:.6f}"
    if pd.notna(row["mae"]):
        line += f", MAE={row['mae']:.6f}"
    if pd.notna(row["r2"]):
        line += f", R2={row['r2']:.6f}"
    print(line)

comparison_sorted.to_csv(DATA_DIR / "wrapper_comparison_results.csv", index=False)
print(f"\nSaved: {DATA_DIR / 'wrapper_comparison_results.csv'}")

=== Wrapper별 최적 성능 (RMSE 기준) ===

RFE + XGBoost: features=5, RMSE=0.100359, MAE=0.074859, R2=0.080899
RFE + Random Forest: features=5, RMSE=0.102251
SFS + Random Forest: features=20, RMSE=0.105945, MAE=0.079905, R2=-0.024257
SFS + XGBoost: features=5, RMSE=0.111340, MAE=0.083110, R2=-0.131235

Saved: ../../../data/processed/wrapper_results/wrapper_comparison_results.csv
